# Module 14: Realtime 2D Human Pose Estimation

Unlike standard object detection that draws a single box around a person, **Pose Estimation** identifies the exact spatial locations of specific human body parts (joints, limbs, head) and maps how they connect.

In this module, we are implementing a Deep Neural Network trained on the **Multi-Person Image Dataset (MPI)**. The network uses **Part Affinity Fields (PAFs)**—a set of 2D vector fields that encode the location and orientation of limbs—to accurately estimate human pose. OpenCV's `dnn` module supports multiple frameworks; here, we will use a pre-trained **Caffe** model.

### 1. Fetching Network Weights and Image Assets
We first download our required workspace assets. This archive contains our target evaluation images (like Tiger Woods swinging a golf club) and the pre-trained neural network weights required to run the PAF model.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from zipfile import ZipFile
from urllib.request import urlretrieve

from IPython.display import YouTubeVideo, display, Image

%matplotlib inline

In [ ]:
def download_and_unzip(url, save_path):
    print(f"Downloading and extracting assests....", end="")

    # Downloading zip file using urllib package.
    urlretrieve(url, save_path)

    try:
        # Extracting zip file using the zipfile package.
        with ZipFile(save_path) as z:
            # Extract ZIP file contents in the same directory.
            z.extractall(os.path.split(save_path)[0])

        print("Done")

    except Exception as e:
        print("\nInvalid file.", e)
URL = r"https://www.dropbox.com/s/089r2yg6aao858l/opencv_bootcamp_assets_NB14.zip?dl=1"

asset_zip_path = os.path.join(os.getcwd(), f"opencv_bootcamp_assets_NB14.zip")

# Download if assest ZIP does not exists. 
if not os.path.exists(asset_zip_path):
    download_and_unzip(URL, asset_zip_path) 

### 2. The Theory: Part Affinity Fields Walkthrough
The mathematics of predicting skeletal structures across multiple people in a single frame is highly complex. The video below is the official project walkthrough detailing how the network groups independent joints into connected limbs without getting confused by overlapping people.

In [ ]:
from IPython.display import HTML, display

# VS Code workaround: HTML button to bypass the iframe security block
html_str = """
<div style="text-align: left; padding: 10px;">
    <a href="https://www.youtube.com/watch?v=RyCsSc_2ZEI" target="_blank" 
       style="background-color: #ff0000; color: white; padding: 10px 20px; text-decoration: none; border-radius: 5px; font-family: sans-serif;">
        ▶️ Open Pose Estimation Video on YouTube
    </a>
</div>
"""
display(HTML(html_str))

### 3. Configuring the Caffe Model & Skeleton Map
A standard OpenCV Caffe model implementation requires two files:
1. **`.prototxt` (Architecture):** The text-based blueprint of the neural network layers.
2. **`.caffemodel` (Weights):** The heavy binary file containing the trained mathematical patterns.

The MPI dataset maps the human body using **15 distinct keypoints** (0 through 14). We define a `POSE_PAIRS` array to tell our rendering engine exactly which keypoints connect to form biological limbs (e.g., mapping point 1 to point 2 creates the right shoulder-to-elbow connection).

In [ ]:
protoFile   = "pose_deploy_linevec_faster_4_stages.prototxt"
weightsFile = os.path.join("model", "pose_iter_160000.caffemodel")
nPoints = 15

POSE_PAIRS = [
    [0, 1],
    [1, 2],
    [2, 3],
    [3, 4],
    [1, 5],
    [5, 6],
    [6, 7],
    [1, 14],
    [14, 8],
    [8, 9],
    [9, 10],
    [14, 11],
    [11, 12],
    [12, 13],
]

net = cv2.dnn.readNetFromCaffe(protoFile, weightsFile)

### 4. Preprocessing the Input Frame
We load our image in standard RGB format and record its original dimensions. 

Neural networks require mathematically standardized inputs. We use `cv2.dnn.blobFromImage` to convert our image matrix into a 4-dimensional blob, resizing it to the specific $368 \times 368$ spatial resolution expected by the Caffe model's input layer, while normalizing pixel intensities to a $[0, 1]$ scale.

In [ ]:
im = cv2.imread("Tiger_Woods_crop.png")
im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
inWidth  = im.shape[1]
inHeight = im.shape[0]

# Wrapped in display() for VS Code to render it properly
display(Image(filename="Tiger_Woods.png"))

In [ ]:
netInputSize = (368, 368)
inpBlob = cv2.dnn.blobFromImage(im, 1.0 / 255, netInputSize, (0, 0, 0), swapRB=True, crop=False)
net.setInput(inpBlob)

### 5. Inference & Probability Heatmaps
We push the preprocessed blob through the network. The output is a massive 4D matrix. 

Instead of returning exact coordinates immediately, the network outputs **Probability Maps (Heatmaps)**. There are 15 distinct heatmaps generated (one for each keypoint). The brighter the color in the heatmap, the more confident the neural network is that the specific joint exists at that spatial location.

In [ ]:
# Forward Pass
output = net.forward()

# Display probability maps
plt.figure(figsize=(20, 5))
for i in range(nPoints):
    probMap = output[0, i, :, :]
    displayMap = cv2.resize(probMap, (inWidth, inHeight), cv2.INTER_LINEAR)
    
    plt.subplot(2, 8, i + 1)
    plt.axis("off")
    plt.imshow(displayMap, cmap="jet")
    
plt.show()  # Required to render the plot in VS Code

### 6. Extracting Exact Coordinates from Heatmaps
To draw our skeleton, we need precise `(X, Y)` pixel coordinates, not blurry heatmaps. 

We iterate through all 15 probability maps and use OpenCV's `cv2.minMaxLoc()` to find the absolute brightest pixel (global maxima) in each map. We then mathematically scale that coordinate up from the network's $368 \times 368$ space back to the physical resolution of our original photograph. If a point's confidence falls below our **10% threshold**, we discard it to prevent drawing ghost limbs.

In [ ]:
# X and Y Scale
scaleX = inWidth  / output.shape[3]
scaleY = inHeight / output.shape[2]

# Empty list to store the detected keypoints
points = []

# Treshold
threshold = 0.1

for i in range(nPoints):
    # Obtain probability map
    probMap = output[0, i, :, :]

    # Find global maxima of the probMap.
    minVal, prob, minLoc, point = cv2.minMaxLoc(probMap)

    # Scale the point to fit on the original image
    x = scaleX * point[0]
    y = scaleY * point[1]

    if prob > threshold:
        # Add the point to the list if the probability is greater than the threshold
        points.append((int(x), int(y)))
    else:
        points.append(None)

### 7. Rendering the Final Skeletal Structure
With our coordinates extracted, we perform two final visual passes:
1. **Joint Overlay:** We draw solid circles at every valid coordinate found in the previous step.
2. **Skeleton Construction:** We iterate through our predefined `POSE_PAIRS` map. If both the start and end joints for a specific bone exist in our valid coordinates list, we draw a connecting line between them.

In [ ]:
imPoints = im.copy()
imSkeleton = im.copy()

# Draw points
for i, p in enumerate(points):
    if p is not None:  # CRITICAL FIX: prevents cv2.circle from crashing on None values
        cv2.circle(imPoints, p, 8, (255, 255, 0), thickness=-1, lineType=cv2.FILLED)
        cv2.putText(imPoints, "{}".format(i), p, cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, lineType=cv2.LINE_AA)

# Draw skeleton
for pair in POSE_PAIRS:
    partA = pair[0]
    partB = pair[1]

    if points[partA] and points[partB]:
        cv2.line(imSkeleton, points[partA], points[partB], (255, 255, 0), 2)
        cv2.circle(imSkeleton, points[partA], 8, (255, 0, 0), thickness=-1, lineType=cv2.FILLED)

plt.figure(figsize=(50, 50))
plt.subplot(121)
plt.axis("off")
plt.imshow(imPoints)
plt.subplot(122)
plt.axis("off")
plt.imshow(imSkeleton)
plt.show()  # Required to render the plot in VS Code

display(Image(filename="Milton_Golf_Swing.png"))